# Forward Modeling with GeoDef

This notebook demonstrates the core forward-modeling workflow:

1. Create a discretized fault
2. Define synthetic GNSS stations
3. Build the Green's matrix
4. Predict displacements from an input slip distribution
5. Visualize the results

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import geodef

# reload modules if they have been changed - helpful during debugging
%load_ext autoreload
%autoreload 2

## 1. Create a fault

We'll model a subduction-zone megathrust: shallow dip, 200 km long, 100 km wide, discretized into a 10x5 grid of patches.

In [ ]:
fault = geodef.Fault.planar(
    lat=-2.0, lon=100.0, depth=25e3,
    strike=320.0, dip=15.0,
    length=200e3, width=100e3,
    n_length=10, n_width=5,
)

print(f"Patches: {fault.n_patches} ({fault.grid_shape[0]} along-strike x {fault.grid_shape[1]} down-dip)")
print(f"Engine:  {fault.engine}")
print(f"Center:  {fault.centers[fault.n_patches // 2]}")

## 2. Define synthetic GNSS stations

Place a grid of stations on the surface above and around the fault.

In [ ]:
sta_lon, sta_lat = np.meshgrid(
    np.linspace(98.5, 101.5, 8),
    np.linspace(-3.5, -0.5, 8),
)
sta_lat = sta_lat.ravel()
sta_lon = sta_lon.ravel()
n_sta = len(sta_lat)

# Create a GNSS dataset (observations are zero — we'll predict them)
gnss = geodef.GNSS(
    sta_lat, sta_lon,
    ve=np.zeros(n_sta), vn=np.zeros(n_sta), vu=np.zeros(n_sta),
    se=np.ones(n_sta), sn=np.ones(n_sta), su=np.ones(n_sta),
)
print(f"Stations: {gnss.n_stations}")

## 3. Build the Green's matrix

`geodef.greens.greens()` computes the raw Green's matrix from the fault, then projects each column through the dataset's `project()` method. For GNSS, this interleaves [E, N, U] per station.

In [ ]:
G = geodef.greens.greens(fault, gnss)
print(f"G shape: {G.shape}")
print(f"  rows = {gnss.n_obs} observations ({gnss.n_stations} stations x 3 components)")
print(f"  cols = {2 * fault.n_patches} (2 slip components x {fault.n_patches} patches)")

## 4. Define a slip distribution and predict displacements

We'll put 3 m of reverse (dip-slip) on the central patches and predict the surface displacement at each station. The slip vector `m` is interleaved: `[ss_0, ds_0, ss_1, ds_1, ...]`.

In [ ]:
# Create slip: 3 m dip-slip on the middle 60% of patches, tapering at edges
nL, nW = fault.grid_shape
slip_dip = np.zeros(fault.n_patches)
slip_strike = np.zeros(fault.n_patches)

# Gaussian-ish slip concentrated in the center
for j in range(nW):
    for i in range(nL):
        k = j * nL + i
        dist_along = (i - nL / 2 + 0.5) / (nL / 2)
        dist_down = (j - nW / 2 + 0.5) / (nW / 2)
        slip_dip[k] = 3.0 * np.exp(-2.0 * (dist_along**2 + dist_down**2))

# Build the interleaved slip vector
m = np.zeros(2 * fault.n_patches)
m[0::2] = slip_strike
m[1::2] = slip_dip

# Predict: d = G @ m
d = G @ m

# Unpack interleaved [e1, n1, u1, e2, n2, u2, ...] back to components
pred_e = d[0::3]
pred_n = d[1::3]
pred_u = d[2::3]

# Also verify via fault.displacement()
ue, un, uz = fault.displacement(sta_lat, sta_lon, slip_strike=slip_strike, slip_dip=slip_dip)

print(f"Max horizontal displacement: {np.max(np.sqrt(pred_e**2 + pred_n**2)):.4f} m")
print(f"Max vertical displacement:   {np.max(np.abs(pred_u)):.4f} m")
print(f"Moment magnitude:            Mw {fault.magnitude(slip_dip):.2f}")
print(f"\nG @ m matches fault.displacement(): {np.allclose(pred_e, ue) and np.allclose(pred_n, un) and np.allclose(pred_u, uz)}")

## 5. Visualize

**Left:** Slip distribution on the fault (patch grid).  
**Right:** Predicted surface displacements (horizontal arrows + vertical color).

In [ ]:
from matplotlib.collections import PolyCollection

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Patch outlines: (N, 5, 2) as [lon, lat]
outlines = fault.patch_outlines

# --- Left panel: slip distribution ---
ax = axes[0]
polys = PolyCollection(
    outlines, array=slip_dip,
    cmap="YlOrRd", edgecolors="k", linewidths=0.5, clim=(0, None),
)
ax.add_collection(polys)
plt.colorbar(polys, ax=ax, label="Dip-slip (m)")
ax.autoscale_view()
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Slip Distribution")
ax.set_aspect("equal")

# --- Right panel: predicted displacements ---
ax = axes[1]

# Fault patches as gray background
polys_bg = PolyCollection(
    outlines, facecolors="lightgray", edgecolors="gray",
    linewidths=0.5, alpha=0.5, zorder=1, label="Fault patches",
)
ax.add_collection(polys_bg)

# Vertical as colored dots
sc = ax.scatter(
    sta_lon, sta_lat, c=pred_u, cmap="RdBu",
    vmin=-np.max(np.abs(pred_u)), vmax=np.max(np.abs(pred_u)),
    s=60, edgecolors="k", linewidths=0.5, zorder=2,
)
plt.colorbar(sc, ax=ax, label="Vertical (m)")

# Horizontal as quiver arrows
scale = 1.0 / np.max(np.sqrt(pred_e**2 + pred_n**2)) * 0.3
ax.quiver(
    sta_lon, sta_lat, pred_e, pred_n,
    scale=1/scale, scale_units="xy", angles="xy",
    color="k", width=0.003, zorder=3,
)

ax.legend(loc="lower right", fontsize=8)
ax.autoscale_view()
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Predicted Surface Displacements")
ax.set_aspect("equal")

plt.tight_layout()
plt.show()

## 6. Joint Green's matrix (GNSS + InSAR)

The same `greens()` function handles multiple data types by vertically stacking the projected blocks.

In [ ]:
# Create an InSAR dataset at the same locations (ascending geometry)
insar = geodef.InSAR(
    sta_lat, sta_lon,
    los=np.zeros(n_sta), sigma=np.ones(n_sta),
    look_e=np.full(n_sta, 0.38),
    look_n=np.full(n_sta, -0.09),
    look_u=np.full(n_sta, 0.92),
)

# Joint Green's matrix
G_joint = geodef.greens.greens(fault, [gnss, insar])
print(f"Joint G shape: {G_joint.shape}")
print(f"  GNSS rows:  {gnss.n_obs}")
print(f"  InSAR rows: {insar.n_obs}")
print(f"  Total:      {gnss.n_obs + insar.n_obs}")

# Predict joint observations
d_joint = G_joint @ m
d_gnss = d_joint[:gnss.n_obs]
d_insar = d_joint[gnss.n_obs:]

# Verify InSAR prediction = look_vector dot displacement
los_pred = 0.38 * pred_e + (-0.09) * pred_n + 0.92 * pred_u
print(f"\nInSAR block matches LOS projection: {np.allclose(d_insar, los_pred)}")